# Order and voice: what does the one-sentence rule follow? (preregistered: `docs/preregistration_order_voice.md`)

The same "which is farther east / north, C or R?" scenes in four renderings: C's sentence last or first, and C's relation
in subject voice ("Leo is 1 unit west of Beth") or object voice ("Beth is 1 unit east of Leo"). This tells apart three
rules that fit the earlier data: C's own step, the direction word in C's sentence, and the last sentence. Behaviour
only, so it is quick.

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. Runtime → Change runtime type → **L4**, **A100** or **H100**. On an L4 part of the 12B model is kept in CPU memory
   (same results, slower: about 1 hour). With an **H100** (or another GPU over 70 GB) the notebook also runs 27B. Then
   **Run all**. About 15–30 minutes for 12B on an A100 or H100, about as much again for 27B.

Results are saved to Drive as they are produced; after a disconnect, just press Run all again. The last cell downloads
`order_voice_results.zip`; send that file back.


In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 / H100 / G4"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 20, f"Only {GB:.0f} GB of GPU memory: pick an L4, A100 or H100"
RUN_27B = GB > 70
# below ~35 GB (e.g. L4) the 12B weights do not all fit: spread them over GPU and CPU memory (same results, slower)
DEV = "" if GB > 35 else "--device auto"
print(f"{GB:.0f} GB: 12B yes{' (part of the model in CPU memory, slower)' if DEV else ''}, 27B {'yes' if RUN_27B else 'no (needs > 70 GB)'}")

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/order_voice_test.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/order", exist_ok=True)
print("already saved:", sorted(os.listdir(f"{RES}/order")))

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Order/voice] Colab run started (or resumed)")

In [ ]:
# 5. Gemma 3 12B
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/order_voice_test.py --model gemma3-12b --batch-size 16 {DEV} 2>&1 | grep -v -i warning

In [ ]:
# 6. Gemma 3 27B (only on a GPU with more than 70 GB)
if RUN_27B:
    !cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/order_voice_test.py --model gemma3-27b --batch-size 8 2>&1 | grep -v -i warning
else:
    print("Skipping 27B: this GPU is too small. 12B alone answers the preregistered question.")

In [ ]:
# 7. One zip of the results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/order_voice_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/order/*"):
        z.write(f, "results/order/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/order_voice_results.zip", f"{D}/order_voice_results.zip")
notify("[Order/voice] zip saved: MyDrive/geomap/order_voice_results.zip")
from google.colab import files
files.download("/content/order_voice_results.zip")